### Chuẩn bị dữ liệu

In [1]:
from google.colab import userdata
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [3]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('datathon-2026-round-1')

print("Path to competition files:", path)

100%|██████████| 27.7M/27.7M [00:01<00:00, 22.5MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/datathon-2026-round-1


In [4]:
folder = path

dfs = {}
files = os.listdir(folder)

for filename in files:
  if filename.endswith('.csv'):
    key = filename.replace('.csv', '')
    print(key)
    dfs[key] = pd.read_csv(os.path.join(folder, filename), low_memory=False)

promotions
products
order_items
orders
reviews
returns
geography
sales
shipments
inventory
web_traffic
sample_submission
customers
payments


In [5]:
# Bảng Master
df_products = dfs['products']
df_customers = dfs['customers']
df_promotions = dfs['promotions']
df_geography = dfs['geography']

# Bảng Transaction
df_order = dfs['orders']
df_order_items = dfs['order_items']
df_payments = dfs['payments']
df_shipments = dfs['shipments']
df_returns = dfs['returns']
df_reviews = dfs['reviews']

# Bảng Analytical
df_sales = dfs['sales']

# Bảng Operational
df_inventory = dfs['inventory']
df_web_traffic = dfs['web_traffic']

In [6]:
df_customers.signup_date = pd.to_datetime(df_customers.signup_date).dt.date
df_promotions.start_date = pd.to_datetime(df_promotions.start_date).dt.date
df_promotions.end_date = pd.to_datetime(df_promotions.end_date).dt.date
df_order.order_date = pd.to_datetime(df_order.order_date).dt.date
df_shipments.ship_date = pd.to_datetime(df_shipments.ship_date).dt.date
df_shipments.delivery_date = pd.to_datetime(df_shipments.delivery_date).dt.date
df_returns.return_date = pd.to_datetime(df_returns.return_date).dt.date
df_reviews.review_date = pd.to_datetime(df_reviews.review_date).dt.date
df_sales.Date = pd.to_datetime(df_sales.Date).dt.date
df_inventory.snapshot_date = pd.to_datetime(df_inventory.snapshot_date).dt.date
df_web_traffic.date = pd.to_datetime(df_web_traffic.date).dt.date

### Trả lời câu hỏi

#### ✅ Câu 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?
C. 144

In [7]:
df_order.order_date = pd.to_datetime(df_order['order_date'])

In [8]:
# Lọc những KH mua > 1 lần
cnt = df_order.customer_id.value_counts()
customer_2more = df_order[df_order['customer_id'].isin(cnt[cnt > 1].index)].copy()

# Sắp xếp theo id
customer_2more = customer_2more.sort_values(by=['customer_id', 'order_date'])
customer_2more.reset_index(drop=True, inplace=True)

# Tính gap
customer_2more['ord_gap'] = customer_2more.groupby('customer_id').order_date.diff()
customer_2more['ord_gap'].median()

Timedelta('144 days 00:00:00')

#### ✅ Câu 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?
Ràng buộc: price > cogs

D. Standard

In [9]:
price = df_products['price']
cogs = df_products['cogs']
df_products['profit'] = (price - cogs) / price
df_products.groupby('segment').profit.mean().sort_values(ascending=False)

,profit
segment,
Standard,0.313442
Premium,0.285377
All-weather,0.284176
Activewear,0.265600
Performance,0.263650
Balanced,0.258038
Trendy,0.240758
Everyday,0.236343


In [10]:
df_products.drop(columns=['profit'], inplace=True)

#### ✅ Câu 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?
B. wrong_size

In [11]:
return_prod = df_returns.merge(df_products, on='product_id')
return_prod[return_prod.category == 'Streetwear'].return_reason.value_counts()

,count
return_reason,
wrong_size,7626
defective,4330
not_as_described,3854
changed_mind,3830
late_delivery,2159


#### ✅ Câu 4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?
C. email_campaign

In [12]:
df_web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

,bounce_rate
traffic_source,
email_campaign,0.004458
social_media,0.004476
paid_search,0.004478
referral,0.004499
organic_search,0.004504
direct,0.004511


#### ✅ Câu 5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?
C. 39%

In [13]:
promo_no_null = df_order_items[df_order_items.promo_id.isna() == False].shape[0]
promo_sum = df_order_items.shape[0]
(promo_no_null/promo_sum)*100

38.663493169565214

#### ✅ Câu 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

A. 55+

In [14]:
df_customers.age_group.isna().sum()

np.int64(0)

In [15]:
#Tính số lượng đơn hàng của mỗi KH
orders_per_customer = df_order.groupby('customer_id').size().reset_index(name='order_count')

#Gộp bảng và tính số đơn trung bình theo age_group
age_merged = orders_per_customer.merge(df_customers[['customer_id', 'age_group']], on='customer_id')
avg_orders_by_age = age_merged.groupby('age_group')['order_count'].mean().reset_index()
idxmax = avg_orders_by_age['order_count'].idxmax()
print("Nhóm tuổi có số đơn hàng trung bình cao nhất: ", avg_orders_by_age['age_group'].loc[idxmax])

Nhóm tuổi có số đơn hàng trung bình cao nhất:  55+


#### ✅ Câu 7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

C. East

In [16]:
#Tính doanh thu từng SP = số lượng * giá
df_order_items['item_revenue'] = df_order_items['quantity'] * df_order_items['unit_price']

#Tính tổng doanh thu mỗi đơn hàng
revenue_per_order = df_order_items.groupby('order_id')['item_revenue'].sum().reset_index()
revenue_per_order.rename(columns={'item_revenue': 'order_revenue'}, inplace=True)

#Nối bảng qua order_id và zip
order_with_revenue = df_order.merge(revenue_per_order, on='order_id', how='left')
order_with_region_revenue = order_with_revenue.merge(df_geography[['zip', 'region']], on='zip', how='left')

#Fill các giá trị NaN
order_with_region_revenue['region'] = order_with_region_revenue['region'].fillna('Unknown')

#Tính toán doanh thu theo khu vực
total_revenue_by_region = order_with_region_revenue.groupby('region')['order_revenue'].sum().reset_index()
idxmax = total_revenue_by_region.loc[total_revenue_by_region['order_revenue'].idxmax()]

print(f"Vùng có tổng doanh thu cao nhất là: {idxmax['region']}")

Vùng có tổng doanh thu cao nhất là: East


In [17]:
df_order_items.drop(columns=['item_revenue'], inplace=True)

#### ✅ Câu 8: Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?
A. credit_card

In [18]:
# Lọc các đơn hàng có order_status là 'cancelled'
cancelled_orders = df_order[df_order['order_status'] == 'cancelled']

# Tìm phương thức thanh toán sử dụng nhiều nhất
top_payment_method = cancelled_orders['payment_method'].value_counts().idxmax()
top_payment_count = cancelled_orders['payment_method'].value_counts().max()

print(f"Phương thức thanh toán phổ biến nhất cho đơn cancelled: {top_payment_method}")

Phương thức thanh toán phổ biến nhất cho đơn cancelled: credit_card


#### ✅ Câu 9: Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?
A. S

In [19]:
# Join order_items với products để đếm số lượng bán ra theo size
df_items_prod = df_order_items.merge(df_products, on='product_id')
size_counts_ordered = df_items_prod['size'].value_counts()

# Join returns với products để đếm số lượng trả về theo size
df_returns_prod = df_returns.merge(df_products, on='product_id')
size_counts_returned = df_returns_prod['size'].value_counts()

# Tính tỷ lệ
return_rates = size_counts_returned / size_counts_ordered

sizes_to_check = ['S', 'M', 'L', 'XL']
valid_sizes = [s for s in sizes_to_check if s in return_rates.index]

highest_return_rate_size = return_rates[valid_sizes].idxmax()
highest_return_rate_val = return_rates[valid_sizes].max()

print(f"Kích thước có tỷ lệ trả hàng cao nhất: {highest_return_rate_size}")

Kích thước có tỷ lệ trả hàng cao nhất: S


#### ✅ Câu 10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?
C. 6 kỳ

In [20]:
# Tính giá trị thanh toán trung bình theo số tháng trả góp (installments)
avg_payment_by_installment = df_payments.groupby('installments')['payment_value'].mean()

# Tìm kỳ hạn có giá trị cao nhất
highest_avg_installment = avg_payment_by_installment.idxmax()
highest_avg_value = avg_payment_by_installment.max()

print(f"Kế hoạch trả góp có giá trị TB cao nhất: {highest_avg_installment} kỳ hạn (Giá trị: {highest_avg_value:,.2f})")

Kế hoạch trả góp có giá trị TB cao nhất: 6 kỳ hạn (Giá trị: 24,446.65)
